# Ders 6: Belirsizlik, Kalibrasyon ve Konformal Tahmin

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 3 (Kayıp fonksiyonları, MLE) ve Ders 5 (Düzenlileştirme).

Bir softmax çıktısı, yalnızca toplamı bire eşit olduğu anlamda bir olasılıktır. Modern ağlar
sistematik olarak **aşırı özgüvenlidir** ve bildirdikleri güven, girdinin eğitim verisine benzeyip
benzemediği hakkında neredeyse hiçbir şey söylemez. Bu defterde iki tür belirsizliği ayırıyor,
standart kestiricileri (Laplace, MC dropout, derin topluluklar) kodluyor, kalibrasyonu düzgün
ölçüyor ve sonlu-örneklem garantisi sunan tek yöntemle — konformal tahminle — bitiriyoruz.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
sigmoid = lambda z: 1/(1+np.exp(-z))
print("Kütüphaneler yüklendi.")


## 1. Rastlantısal (Aleatoric) ve Bilgisel (Epistemic) Belirsizlik

- **Rastlantısal** belirsizlik, veri üretme sürecindeki gürültüdür. Veri arttıkça küçülmez — aynı
  $x$'i iki kez ölçün, farklı $y$ elde edersiniz.
- **Bilgisel** belirsizlik, model hakkındaki bilgisizliktir. Veriyle birlikte küçülür ve dağılım
  dışında patlaması gerekir.

Bu ayrım akademik bir incelik değildir: rastlantısal belirsizlik size veri toplamayı bırakmanızı,
bilgisel belirsizlik ise nerede toplamanız gerektiğini söyler. Aşağıda, heteroskedastik olabilirlikli
$p(y \mid x) = \mathcal{N}(\mu_\theta(x), \sigma^2_\theta(x))$ bir model, ortasında boşluk olan bir
veriye uyduruluyor.


In [ ]:
def f_true(x):  return np.sin(2*x) + 0.3*x
def noise(x):   return 0.05 + 0.35*np.abs(np.sin(1.5*x))     # x'e bağlı (rastlantısal) gürültü

rng = np.random.default_rng(1)
x_tr = np.concatenate([rng.uniform(-3, -1, 60), rng.uniform(1, 3, 60)])   # [-1, 1] aralığındaki boşluğa dikkat
y_tr = f_true(x_tr) + noise(x_tr)*rng.normal(size=x_tr.size)
x_te = np.linspace(-5, 5, 400)

# Rastgele Fourier öznitelikleri üzerinde Bayesçi doğrusal regresyon iki terimi de kapalı formda verir
def features(x, D=40, ls=0.7, seed=0):
    r = np.random.default_rng(seed)
    W, b = r.normal(0, 1/ls, D), r.uniform(0, 2*np.pi, D)
    return np.sqrt(2/D)*np.cos(np.outer(x, W) + b)

Phi_tr, Phi_te = features(x_tr), features(x_te)
sig2, alpha = 0.09, 1.0
A = Phi_tr.T@Phi_tr/sig2 + alpha*np.eye(Phi_tr.shape[1])
Sigma = np.linalg.inv(A)
m = Sigma @ Phi_tr.T @ y_tr / sig2

mu = Phi_te @ m
var_epi = np.sum(Phi_te @ Sigma * Phi_te, axis=1)      # veriyle küçülür, destek dışında patlar
var_ale = noise(x_te)**2                               # indirgenemez

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (name, v) in zip(axes, [("yalnızca bilgisel", var_epi), ("yalnızca rastlantısal", var_ale),
                                ("toplam = bilgisel + rastlantısal", var_epi + var_ale)]):
    s = np.sqrt(v)
    ax.fill_between(x_te, mu-2*s, mu+2*s, alpha=0.3, color="steelblue", label="+/- 2 std")
    ax.plot(x_te, f_true(x_te), "k--", lw=1.2, label="gerçek")
    ax.plot(x_te, mu, lw=2, c="crimson", label="tahmin")
    ax.scatter(x_tr, y_tr, s=10, c="k", alpha=0.5, zorder=3)
    ax.axvspan(-1, 1, color="orange", alpha=0.12)
    ax.set_ylim(-3, 3); ax.set_title(name); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("Taralı bant = eğitim verisi olmayan bölge", fontsize=12)
plt.tight_layout(); plt.show()

print("Veri boşluğunda bilgisel varyans   :", round(float(var_epi[(x_te>-1)&(x_te<1)].mean()), 4))
print("Verinin bulunduğu yerde bilgisel varyans:", round(float(var_epi[(x_te>1.2)&(x_te<2.8)].mean()), 4))


## 2. Pratik Kestiriciler: Laplace, MC Dropout, Derin Topluluklar

Ağ ağırlıkları üzerinde tam Bayesçi çıkarım hesaplanamaz olduğundan üç yaklaşım baskındır:

**Laplace yaklaşımı.** Önce MAP çözümü bulunur, sonra eğrilik kullanılarak etrafına bir Gauss
oturtulur: $p(\theta \mid \mathcal{D}) \approx \mathcal{N}(\theta^\star, H^{-1})$. Ucuzdur, sonradan
(post-hoc) uygulanabilir ve zaten eğitilmiş bir ağa takılabilir — genellikle yalnızca son katmana.

**MC dropout.** Test sırasında dropout açık bırakılır ve $T$ stokastik ileri geçiş ortalanır. Bu,
Bernoulli sonsalıyla varyasyonel çıkarımdır; neredeyse bedavadır ama bildirdiği belirsizlik veriyle
değil dropout oranıyla sınırlıdır.

**Derin topluluklar.** Farklı ilk değerlerden $M$ ağ eğitilip ortalanır. Üçünün içinde tutarlı
biçimde en güçlüsüdür; çünkü bağımsız koşular gerçekten farklı havzalara iner ve veri olmayan yerde
anlamlı biçimde birbirinden ayrışır.


In [ ]:
# Lojistik regresyon: tam sonsal (MCMC'siz ızgara) ile Laplace yaklaşımı
rng = np.random.default_rng(2)
n = 40
X = np.c_[np.ones(n), rng.normal(size=n)]
w_true = np.array([0.3, 2.5])
y = (rng.random(n) < sigmoid(X@w_true)).astype(float)

def nll(w):  
    z = X@w
    return -np.sum(y*z - np.logaddexp(0, z)) + 0.5*0.1*np.sum(w**2)

# Newton yöntemiyle MAP
w = np.zeros(2)
for _ in range(50):
    p = sigmoid(X@w)
    g = X.T@(p-y) + 0.1*w
    H = X.T@(X*(p*(1-p))[:, None]) + 0.1*np.eye(2)
    w -= np.linalg.solve(H, g)
w_map, H_map = w, H
Sig_map = np.linalg.inv(H_map)

g1 = np.linspace(w_map[0]-3, w_map[0]+3, 160)
g2 = np.linspace(w_map[1]-4, w_map[1]+4, 160)
GG1, GG2 = np.meshgrid(g1, g2)
post = np.exp(-np.array([[nll(np.array([a, b])) for a in g1] for b in g2]))
post /= post.sum()
lap = np.exp(-0.5*np.einsum("ijk,kl,ijl->ij",
        np.stack([GG1-w_map[0], GG2-w_map[1]], -1), H_map,
        np.stack([GG1-w_map[0], GG2-w_map[1]], -1)))
lap /= lap.sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].contour(GG1, GG2, post, 8, cmap="Blues")
axes[0].contour(GG1, GG2, lap, 8, cmap="Reds", linestyles="--")
axes[0].scatter(*w_map, c="k", s=50, zorder=4)
axes[0].set_xlabel("w0"); axes[0].set_ylabel("w1")
axes[0].set_title("Tam sonsal (mavi) ve Laplace (kırmızı kesikli)")

xs = np.linspace(-4, 4, 200)
Xs = np.c_[np.ones_like(xs), xs]
samples = rng.multivariate_normal(w_map, Sig_map, 300)
probs = sigmoid(Xs @ samples.T)
axes[1].plot(xs, sigmoid(Xs@w_map), lw=2, c="crimson", label="MAP (nokta kestirimi)")
axes[1].plot(xs, probs.mean(1), lw=2, c="steelblue", label="sonsal öngörü dağılımı")
axes[1].fill_between(xs, np.percentile(probs, 5, axis=1), np.percentile(probs, 95, axis=1),
                     alpha=0.25, color="steelblue")
axes[1].scatter(X[:, 1], y, s=14, c="k", alpha=0.6)
axes[1].set_title("Sonsal üzerinden ortalama almak kararı yumuşatır")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Topluluk ve MC dropout: asıl önemli çeşitlilik HAVZALAR arasındadır
def toy_fit(seed, drop=False, drop_seed=0):
    r = np.random.default_rng(drop_seed)
    Phi = features(x_tr, D=40, seed=seed)                       # farklı rastgele öznitelikler = farklı model
    if drop:
        Phi = Phi*(r.random(Phi.shape) > 0.2)/0.8
    A = Phi.T@Phi + 1.0*np.eye(Phi.shape[1])
    mm = np.linalg.solve(A, Phi.T@y_tr)
    return features(x_te, D=40, seed=seed) @ mm

ens = np.stack([toy_fit(s) for s in range(12)])
drp = np.stack([toy_fit(0, drop=True, drop_seed=100+i) for i in range(12)])
axes[2].plot(x_te, ens.T, lw=0.8, c="steelblue", alpha=0.7)
axes[2].plot(x_te, drp.T, lw=0.8, c="orange", alpha=0.7)
axes[2].plot([], [], c="steelblue", label="derin topluluk üyeleri")
axes[2].plot([], [], c="orange", label="MC dropout örnekleri")
axes[2].scatter(x_tr, y_tr, s=10, c="k", alpha=0.5, zorder=3)
axes[2].set_ylim(-3, 3); axes[2].axvspan(-1, 1, color="orange", alpha=0.12)
axes[2].set_title("Destek dışında anlaşmazlık"); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"veri boşluğundaki yayılım: topluluk {ens[:, (x_te>-1)&(x_te<1)].std(0).mean():.3f}  "
      f"| dropout {drp[:, (x_te>-1)&(x_te<1)].std(0).mean():.3f}")


## 3. Kalibrasyon ve Ölçülmesi

Bir sınıflandırıcı, $p$ güveniyle yaptığı tahminlerin $p$ oranı doğru çıkıyorsa **kalibredir**.
Çapraz entropiyle yakınsamaya kadar eğitilen derin ağlar genellikle kalibre değildir: doğruluk
iyileşmeyi bıraktıktan çok sonra bile kayıp, daha büyük logitleri ödüllendirmeye devam eder.

**Beklenen Kalibrasyon Hatası (ECE)** tahminleri güvene göre kutulara ayırır ve farkı ortalar:

$$\text{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{n}\,\big|\,\text{doğruluk}(B_b) - \text{güven}(B_b)\,\big| .$$

ECE bir *teşhis aracıdır*, bir hedef değil: kutulama şemasına duyarlıdır ve her girdi için taban
oranı tahmin eden bir model sıfır ECE'ye sahiptir ama hiçbir işe yaramaz.


In [ ]:
def ece(conf, correct, n_bins=12):
    edges = np.linspace(0, 1, n_bins+1)
    e, accs, confs, ws = 0.0, [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            accs.append(np.nan); confs.append(np.nan); ws.append(0); continue
        a, c, w = correct[m].mean(), conf[m].mean(), m.mean()
        e += w*abs(a-c); accs.append(a); confs.append(c); ws.append(w)
    return e, np.array(accs), np.array(confs), np.array(ws), edges

rng = np.random.default_rng(4)
n, K = 4000, 10
true_p = rng.dirichlet(np.ones(K)*0.4, n)
y = np.array([rng.choice(K, p=p) for p in true_p])
logits = np.log(true_p + 1e-12)

def evaluate(logit_scale, label):
    z = logits*logit_scale
    p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    conf, pred = p.max(1), p.argmax(1)
    e, accs, confs, ws, edges = ece(conf, (pred == y).astype(float))
    return dict(label=label, ece=e, accs=accs, confs=confs, ws=ws, edges=edges,
                nll=-np.log(p[np.arange(n), y] + 1e-12).mean(), acc=(pred == y).mean())

overconf = evaluate(2.2, "aşırı özgüvenli (T<1)")
calib    = evaluate(1.0, "kalibre")
underconf= evaluate(0.5, "özgüvensiz (T>1)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
centers = (calib["edges"][:-1] + calib["edges"][1:])/2
for ax, r in zip(axes, [overconf, calib, underconf]):
    ax.bar(centers, np.nan_to_num(r["accs"]), width=1/12*0.9, alpha=0.8, label="doğruluk")
    ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="kusursuz kalibrasyon")
    ax.plot(centers, r["confs"], "o", c="crimson", ms=5, label="ortalama güven")
    ax.set_xlabel("güven"); ax.set_ylabel("doğruluk")
    ax.set_title(f"{r['label']}\nECE={r['ece']:.3f}  NLL={r['nll']:.3f}  doğruluk={r['acc']:.3f}", fontsize=11)
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.suptitle("Güvenilirlik diyagramları -- üç panelde de doğruluk aynı", fontsize=12)
plt.tight_layout(); plt.show()


### Sıcaklık ölçekleme

En ucuz çözüm: logitleri, tutulan bir doğrulama kümesinde NLL minimize edilerek bulunan tek bir skaler
$T$'ye bölmek. Sınıf sıralamasını değiştiremediği için **doğruluk kesinlikle değişmez** — yalnızca
güveni yeniden ölçekler. Bir parametre, bir doğrulama kümesi ve standart bir ağın kalibrasyon
bozukluğunun çoğu ortadan kalkar.


In [ ]:
z_over = logits*2.2
split = n//2
val, test = slice(0, split), slice(split, n)

def nll_at(T, sl):
    z = z_over[sl]/T
    p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    return -np.log(p[np.arange(len(p)), y[sl]] + 1e-12).mean()

Ts = np.linspace(0.5, 6, 120)
nlls = [nll_at(T, val) for T in Ts]
T_star = Ts[int(np.argmin(nlls))]

def stats(T):
    z = z_over[test]/T
    p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    conf, pred = p.max(1), p.argmax(1)
    e, *_ = ece(conf, (pred == y[test]).astype(float))
    return e, -np.log(p[np.arange(len(p)), y[test]] + 1e-12).mean(), (pred == y[test]).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(Ts, nlls, lw=2)
axes[0].axvline(T_star, c="crimson", ls="--", label=f"T* = {T_star:.2f}")
axes[0].set_xlabel("sıcaklık T"); axes[0].set_ylabel("doğrulama NLL")
axes[0].set_title("Tutulan veride tek bir skaler uydurmak"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

for T, c, lab in [(1.0, "crimson", "önce (T=1)"), (T_star, "seagreen", f"sonra (T={T_star:.2f})")]:
    z = z_over[test]/T
    p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    axes[1].hist(p.max(1), bins=40, alpha=0.55, color=c, label=lab)
axes[1].set_xlabel("güven"); axes[1].set_ylabel("sayı")
axes[1].set_title("Sıcaklık ölçekleme yalnızca güveni değiştirir"); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

for T, name in [(1.0, "önce"), (T_star, "sonra")]:
    e, nl, ac = stats(T)
    print(f"{name}: ECE {e:.4f}   NLL {nl:.4f}   doğruluk {ac:.4f}")
print("Doğruluk bit düzeyinde aynı: monoton bir ölçekleme sınıfların sırasını değiştiremez.")


## 4. Konformal Tahmin: Model Varsayımı Olmadan Garanti

Yukarıdakilerin hepsi belirsizliği kestirir; hiçbiri bir şey *garanti etmez*. **Bölünmüş konformal
tahmin** eder — tek bir varsayım altında: kalibrasyon ve test verisinin değiştirilebilir
(exchangeable) olması.

1. Modelin hiç görmediği bir kalibrasyon kümesi ayırın.
2. Her kalibrasyon noktası için bir uyumsuzluk skoru $s_i$ hesaplayın (ör. $1 - \hat p(y_i \mid x_i)$).
3. $\hat q$, bu skorların $\lceil (n+1)(1-\alpha)\rceil / n$ ampirik kuantili olsun.
4. **Küme** tahminini verin: $\{y : s(x, y) \le \hat q\}$.

O zaman $\mathbb{P}(y_{\text{test}} \in \mathcal{C}(x_{\text{test}})) \ge 1-\alpha$ olur — herhangi
bir model, herhangi bir dağılım ve sonlu $n$ için. Model kalitesi *geçerliliği* etkilemez; yalnızca
kümelerin boyutunu etkiler. Zayıf bir model de %90 kapsama ulaşır; sadece bunu daha büyük kümeler
üreterek yapmak zorundadır.


In [ ]:
alpha = 0.1
rng = np.random.default_rng(7)
n_cal, n_test, K = 1000, 2000, 10

# quality=1 -> güçlü model; quality<1 -> aynı hedefler, çok daha zayıf bir tahminci
def make(n, quality, conc=0.3):
    tp = rng.dirichlet(np.ones(K)*conc, n)
    yy = np.array([rng.choice(K, p=p) for p in tp])
    pp = quality*tp + (1-quality)*rng.dirichlet(np.ones(K), n)
    return pp/pp.sum(1, keepdims=True), yy

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
report = []
for quality, name, ax in [(1.0, "güçlü model", axes[0]), (0.35, "zayıf model", axes[1])]:
    p_cal, y_cal = make(n_cal, quality)
    p_te,  y_te  = make(n_test, quality)
    s_cal = 1 - p_cal[np.arange(n_cal), y_cal]
    q = np.quantile(s_cal, np.ceil((n_cal+1)*(1-alpha))/n_cal, method="higher")
    sets = (1 - p_te) <= q
    cov  = sets[np.arange(n_test), y_te].mean()
    size = sets.sum(1)
    acc  = (p_te.argmax(1) == y_te).mean()
    ax.hist(size, bins=np.arange(0, K+2)-0.5, color="steelblue", alpha=0.85)
    ax.set_xlabel("tahmin kümesi boyutu"); ax.set_ylabel("sayı"); ax.set_xlim(-0.5, K+0.5)
    ax.set_title(f"{name} (ilk-1 doğruluk {acc:.2f})\nkapsama {cov:.3f} (hedef {1-alpha:.2f}), "
                 f"ortalama boyut {size.mean():.2f}", fontsize=11)
    report.append((name, cov, size.mean(), q))

# Kapsama ortalamada tamdır ama koşudan koşuya O(1/sqrt(n_cal)) kadar değişir
covs = []
for _ in range(200):
    p_cal, y_cal = make(300, 1.0); p_te, y_te = make(500, 1.0)
    s = 1 - p_cal[np.arange(300), y_cal]
    q = np.quantile(s, np.ceil(301*(1-alpha))/300, method="higher")
    covs.append(((1-p_te) <= q)[np.arange(500), y_te].mean())
axes[2].hist(covs, bins=25, color="seagreen", alpha=0.85)
axes[2].axvline(1-alpha, c="crimson", lw=2, label=f"hedef {1-alpha}")
axes[2].axvline(np.mean(covs), c="k", ls="--", lw=2, label=f"ortalama {np.mean(covs):.3f}")
axes[2].set_xlabel("200 tekrarda ampirik kapsama"); axes[2].set_ylabel("sayı")
axes[2].set_title("Marjinal kapsama ortalamada sağlanır"); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()
for name, cov, size, q in report:
    print(f"{name:24s} kapsama {cov:.3f}   ortalama küme boyutu {size:.2f}   eşik q = {q:.3f}")
print("\nİki model de hedef kapsamaya ulaşıyor. Zayıf olan bedelini çok daha büyük kümelerle ödüyor.")


Açıkça belirtilmesi gereken iki sınırlama var. Garanti **marjinaldir**, yani popülasyon üzerinden
ortalamadır — $x$'e koşullu değildir; dolayısıyla belirli bir altgrup sistematik olarak eksik
kapsanabilir (APS ve RAPS gibi uyarlanır skorlar bunu iyileştirir). Ve değiştirilebilirlik dağılım
kayması altında bozulur; tam da belirsizliğin en çok önemli olduğu durumda.

## 5. Uygun Skorlama Kuralları

Bir skorlama kuralı, gerçek olasılıklar bildirildiğinde minimum oluyorsa **uygundur** (proper);
dolayısıyla güveni çarpıtarak kandırılamaz. NLL ve Brier uygundur; doğruluk değildir — bu yüzden bir
model doğrulukta iyileşirken olasılıkları kötüleşebilir.

Brier skoru yorumlanabilir parçalara ayrışır:

$$\text{Brier} = \underbrace{\mathbb{E}(\hat p - \bar y_{\hat p})^2}_{\text{kalibrasyon}}
- \underbrace{\mathbb{E}(\bar y_{\hat p} - \bar y)^2}_{\text{ayırt edicilik}}
+ \underbrace{\bar y (1-\bar y)}_{\text{indirgenemez}} .$$

Düşük kalibrasyon hatası **ve** yüksek ayırt edicilik istenir. Her seferinde taban oranı tahmin etmek
kusursuz kalibredir ve tamamen yararsızdır — bunun iyi skor almasını engelleyen şey ayırt ediciliktir.


In [ ]:
rng = np.random.default_rng(11)
n = 4000
p_true = rng.beta(1.6, 1.6, n)
y = (rng.random(n) < p_true).astype(float)

models = {
    "gerçek olasılıklar":  p_true,
    "aşırı özgüvenli":       np.clip((p_true-0.5)*2.2 + 0.5, 0.001, 0.999),
    "özgüvensiz":      (p_true-0.5)*0.4 + 0.5,
    "yalnızca taban oran":      np.full(n, y.mean()),
}

def brier_decomp(p, y, bins=15):
    edges = np.linspace(0, 1, bins+1); idx = np.clip(np.digitize(p, edges)-1, 0, bins-1)
    cal = res = 0.0
    for b in range(bins):
        m = idx == b
        if m.sum() == 0: continue
        w = m.mean(); cal += w*(p[m].mean() - y[m].mean())**2; res += w*(y[m].mean() - y.mean())**2
    return cal, res, y.mean()*(1-y.mean())

print(f"{'model':22s} {'Brier':>8} {'NLL':>8} {'kalib':>8} {'ayırtE':>8} {'doğruluk':>9}")
rows = []
for name, p in models.items():
    br = np.mean((p-y)**2)
    nl = -np.mean(y*np.log(p+1e-12) + (1-y)*np.log(1-p+1e-12))
    cal, res, unc = brier_decomp(p, y)
    acc = np.mean((p > 0.5) == (y > 0.5))
    rows.append((name, br, nl, cal, res, acc))
    print(f"{name:22s} {br:8.4f} {nl:8.4f} {cal:8.4f} {res:8.4f} {acc:9.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
names = [r[0] for r in rows]
axes[0].barh(names, [r[3] for r in rows], label="kalibrasyon (küçük iyi)")
axes[0].barh(names, [-r[4] for r in rows], label="-ayırt edicilik (küçük iyi)")
axes[0].axvline(0, c="k", lw=1); axes[0].legend(fontsize=9)
axes[0].set_title("Brier ayrışımı")

for name, p in models.items():
    axes[1].plot(np.sort(p), np.linspace(0, 1, n), lw=2, label=name)
axes[1].set_xlabel("tahmin edilen olasılık"); axes[1].set_ylabel("ampirik dağılım fonksiyonu")
axes[1].set_title("Bildirilen olasılıkların dağılımı"); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("\n'yalnızca taban oran': kalibrasyon hatası ~0, ayırt edicilik 0 -- kalibre ve işe yaramaz.")


## 6. Özet

| Kavram | Açıklama |
|---|---|
| **Rastlantısal** | Veri gürültüsü; veriyle küçülmez |
| **Bilgisel** | Model bilgisizliği; veriyle küçülür, dağılım dışında büyümelidir |
| **Laplace** | MAP etrafında Hessian'la kurulan Gauss; ucuz ve sonradan uygulanabilir |
| **MC dropout** | Test zamanı dropout = varyasyonel çıkarım; ucuz ama sınırlı |
| **Derin topluluk** | Bağımsız koşular anlamlı biçimde ayrışır; en güçlü basit temel yöntem |
| **ECE** | Kutulanmış $\lvert$doğruluk $-$ güven$\rvert$; bir teşhis, hedef değil |
| **Sıcaklık ölçekleme** | Logitlerde tek skaler; güveni düzeltir, doğruluğu asla değiştirmez |
| **Konformal tahmin** | Değiştirilebilirlik altında dağılımdan bağımsız sonlu-örneklem kapsaması |
| **Marjinal ve koşullu** | Konformal kapsama ortalamada geçerlidir, altgrup başına değil |
| **Uygun skorlama** | NLL ve Brier kandırılamaz; doğruluk kandırılabilir |
| **Brier ayrışımı** | kalibrasyon $-$ ayırt edicilik $+$ indirgenemez |

**Sonraki Defter →** Difüzyon ve Skor Tabanlı Üretici Modeller
